# Configuración Inicial y Carga de Directorios

In [ ]:
# ===================== CV sujeto-wise desde carpetas dataset_v3 =====================
import os, re, gc, numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, accuracy_score, roc_curve, auc
from sklearn.preprocessing import label_binarize
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, regularizers, backend as K



In [ ]:
BASE_DIR = Path("../database/dataset_v3_Augmented_with_CLAHE")     
ORDER    = ['normal','benigno','maligno']  
COLOR_MODE  = 'grayscale'         
TARGET_SIZE = (224, 224)
BATCH_SIZE  = 32
R_SEED = 2025
np.random.seed(R_SEED); tf.random.set_seed(R_SEED)

# Prevención de Fuga de Datos: Extracción de Pacientes

In [ ]:
# -------- 1) Reconstruir DF desde disco (usamos TODAS las imágenes de train+val+test) --------
def parse_patient_from_fname(fname: str) -> str:
    """
    Extrae el 'patient' desde nombre tipo: SOURCE_PATIENT_ORIGINAL.ext
    donde PATIENT ya incluye un '_' (p.ej., CBIS_123, MIAS_mdb003).
    """
    stem = Path(fname).stem
    toks = stem.split('_')
    if len(toks) >= 3:
        return toks[1] + "_" + toks[2]
    # fallback: si por alguna razón no cumple el patrón
    return toks[1] if len(toks) > 1 else stem

In [ ]:
rows = []
for split in ["train", "val", "test"]:
    for label in ORDER:
        d = BASE_DIR / split / label
        if not d.exists(): 
            continue
        for fname in os.listdir(d):
            fpath = d / fname
            if not fpath.is_file():
                continue
            rows.append({
                "filepath": str(fpath.resolve()),
                "label_name": label,
                "patient": parse_patient_from_fname(fname)
            })

In [ ]:
df_all = pd.DataFrame(rows)
assert len(df_all) > 0, "No se encontraron imágenes."

# (Opcional) quitar duplicados exactos por ruta si existieran
df_all = df_all.drop_duplicates(subset=["filepath"]).reset_index(drop=True)

# Índices de clase
label2idx = {'normal':0,'benigno':1,'maligno':2}
df_all["label_idx"] = df_all["label_name"].map(label2idx)

print("Total imgs:", len(df_all))
print("Clases:\n", df_all["label_name"].value_counts())
print("Pacientes únicos:", df_all["patient"].nunique())

# Pipeline de Aumento de Datos (Data Augmentation)

In [ ]:
# -------- 2) Generadores (augment SOLO en train) --------
train_aug = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20, zoom_range=0.20,
    width_shift_range=0.05, height_shift_range=0.05,
    horizontal_flip=True, fill_mode='nearest'
)
plain = ImageDataGenerator(rescale=1./255)

In [ ]:
def make_gens_df(df_train, df_val, df_test, seed=1):
    train_gen = train_aug.flow_from_dataframe(
        df_train, x_col='filepath', y_col='label_name',
        target_size=TARGET_SIZE, color_mode=COLOR_MODE,
        class_mode='categorical', batch_size=BATCH_SIZE,
        shuffle=True, seed=seed, classes=ORDER
    )
    val_gen = plain.flow_from_dataframe(
        df_val, x_col='filepath', y_col='label_name',
        target_size=TARGET_SIZE, color_mode=COLOR_MODE,
        class_mode='categorical', batch_size=BATCH_SIZE,
        shuffle=False, classes=ORDER
    )
    test_gen = plain.flow_from_dataframe(
        df_test, x_col='filepath', y_col='label_name',
        target_size=TARGET_SIZE, color_mode=COLOR_MODE,
        class_mode='categorical', batch_size=BATCH_SIZE,
        shuffle=False, classes=ORDER
    )
    # mapping consistente
    assert train_gen.class_indices == {'normal':0,'benigno':1,'maligno':2}, train_gen.class_indices
    assert val_gen.class_indices   == train_gen.class_indices
    assert test_gen.class_indices  == train_gen.class_indices
    return train_gen, val_gen, test_gen

# Balanceo de Clases: Oversampling Dinámico

In [ ]:
# -------- 3) Oversampling moderado de 'normal' SOLO en train del fold --------
def oversample_normal(df_train, target_max=650, mult=5, label_col='label_name', seed=R_SEED):
    """
    Sube 'normal' hasta min(target_max, mult * n_normal_fold).
    No toca benigno/maligno. Devuelve df mezclado.
    """
    df = df_train.copy()
    n_norm = (df[label_col] == 'normal').sum()
    target = min(target_max, n_norm * mult)
    need   = max(0, target - n_norm)
    if need > 0 and n_norm > 0:
        extra = df[df[label_col]=='normal'].sample(n=need, replace=True, random_state=seed)
        df = pd.concat([df, extra], ignore_index=True)
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)

# Arquitectura del Modelo CNN (Sequential)

In [ ]:
# -------- 4) Modelo (puedes sustituir por tu VGG/ResNet FT) --------
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers

def build_model(input_shape=(224, 224, 1), num_classes=3, l2=0.0001):
    model = models.Sequential()

    # Primera capa convolucional
    model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape, padding='same'))
    model.add(layers.MaxPooling2D((2, 2)))

    # Segunda capa convolucional
    model.add(layers.Conv2D(64, (3, 3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D((2, 2)))

    # Tercera capa convolucional
    model.add(layers.Conv2D(128, (3, 3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D((2, 2)))

    # Cuarta capa convolucional
    model.add(layers.Conv2D(256, (3, 3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D((2, 2)))

    # Quinta capa convolucional
    model.add(layers.Conv2D(512, (3, 3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D((2, 2)))

    # Aplanar para pasar a las capas densas
    model.add(layers.Flatten())

    # Capa completamente conectada con 1024 neuronas y regularización más ligera
    model.add(layers.Dense(1024, activation='relu', kernel_regularizer=regularizers.l2(l2)))
    model.add(layers.Dropout(0.3))  # Dropout más ligero para mantener más información

    # Capa de salida para 3 clases
    model.add(layers.Dense(num_classes, activation='softmax'))

    # Compilación original del K-Fold
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
                  metrics=['accuracy'])
    
    return model

# Validación Cruzada de 5 Pliegues (K-Fold Sujeto-Wise)

In [ ]:
# -------- 5) 5-fold CV sujeto-wise + estratificado --------
outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=R_SEED)
y_all = df_all["label_idx"].values
g_all = df_all["patient"].values

results = []

In [ ]:
for fold, (trval_idx, te_idx) in enumerate(outer.split(df_all, y=y_all, groups=g_all), 1):
    print(f"\n========== Fold {fold}/5 ==========")
    df_tv = df_all.iloc[trval_idx].reset_index(drop=True)
    df_te = df_all.iloc[te_idx].reset_index(drop=True)

    # split interno para validación (también sujeto-wise + estratificado)
    inner = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=R_SEED+fold)
    y_tv  = df_tv["label_idx"].values
    g_tv  = df_tv["patient"].values
    tr_idx, va_idx = next(inner.split(df_tv, y=y_tv, groups=g_tv))
    df_tr = df_tv.iloc[tr_idx].reset_index(drop=True)
    df_va = df_tv.iloc[va_idx].reset_index(drop=True)

    # checks sujeto-wise
    assert set(df_tr["patient"]).isdisjoint(set(df_va["patient"]))
    assert set(df_tr["patient"]).isdisjoint(set(df_te["patient"]))
    assert set(df_va["patient"]).isdisjoint(set(df_te["patient"]))

    # oversampling moderado SOLO para 'normal' en train del fold
    df_tr_os = oversample_normal(df_tr, target_max=650, mult=5)

    # class_weight desde el TRAIN ORIGINAL del fold (antes de oversampling)
    classes = np.array([0,1,2])
    cw_vals = compute_class_weight(class_weight='balanced', classes=classes, y=df_tr['label_idx'].values)
    class_weight = dict(zip(classes, cw_vals))

    # generadores
    train_gen, val_gen, test_gen = make_gens_df(df_tr_os, df_va, df_te, seed=fold)

    # modelo
    in_ch = 1 if COLOR_MODE=='grayscale' else 3
    model = build_model(input_shape=(TARGET_SIZE[0], TARGET_SIZE[1], in_ch))

    cbs = [
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3),
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)
    ]

    # entrenar (selección en validación)
    history = model.fit(
        train_gen, epochs=60,
        validation_data=val_gen,
        class_weight=class_weight,
        callbacks=cbs, verbose=1
    )

    # evaluar en TEST del fold
    test_loss, test_acc = model.evaluate(test_gen, verbose=0)
    print(f"[Fold {fold}] Test acc: {test_acc:.4f}")

    # métricas clínicas y AUC macro
    y_true = df_te['label_name'].map(label2idx).values
    y_prob = model.predict(test_gen, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)

    cm = confusion_matrix(y_true, y_pred, labels=[0,1,2])
    total = cm.sum()
    def sens_spec(cm, cls):
        TP = cm[cls, cls]
        FN = cm[cls, :].sum() - TP
        FP = cm[:, cls].sum() - TP
        TN = total - (TP + FN + FP)
        sens = TP/(TP+FN) if (TP+FN)>0 else np.nan
        spec = TN/(TN+FP) if (TN+FP)>0 else np.nan
        return sens, spec

    sensN, specN = sens_spec(cm, 0)
    sensB, specB = sens_spec(cm, 1)
    sensM, specM = sens_spec(cm, 2)

    y_true_bin = label_binarize(y_true, classes=[0,1,2])
    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(3):
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(3)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(3): mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= 3.0
    macroAUC = auc(all_fpr, mean_tpr)

    results.append({
        'fold': fold, 'acc': float(test_acc),
        'sensN': sensN, 'specN': specN,
        'sensB': sensB, 'specB': specB,
        'sensM': sensM, 'specM': specM,
        'macroAUC': macroAUC
    })

    # limpieza memoria
    K.clear_session(); del model
    gc.collect()


# Resultados y  Métricas

In [ ]:
# -------- 6) Resumen final: mean ± std --------
res = pd.DataFrame(results).sort_values('fold')
print("\nResultados por fold:\n", res.round(4))
print("\nPromedios (mean ± std):")
for k in ['acc','macroAUC','sensN','specN','sensB','specB','sensM','specM']:
    print(f"{k:>8s}: {res[k].mean():.3f} ± {res[k].std():.3f}")


Total imgs: 4320
Clases:
 label_name
benigno    2280
maligno    1317
normal      723
Name: count, dtype: int64
Pacientes únicos: 1200

========== Fold 1/5 ==========
Found 2981 validated image filenames belonging to 3 classes.
Found 686 validated image filenames belonging to 3 classes.
Found 845 validated image filenames belonging to 3 classes.
Epoch 1/60
94/94 [==============================] - 60s 469ms/step - loss: 1.4843 - accuracy: 0.5502 - val_loss: 1.8905 - val_accuracy: 0.4038 - lr: 0.0010
Epoch 2/60
94/94 [==============================] - 37s 393ms/step - loss: 1.2851 - accuracy: 0.5606 - val_loss: 1.8533 - val_accuracy: 0.3149 - lr: 0.0010
Epoch 3/60
94/94 [==============================] - 39s 409ms/step - loss: 1.1958 - accuracy: 0.5629 - val_loss: 1.3886 - val_accuracy: 0.5845 - lr: 0.0010
Epoch 4/60
94/94 [==============================] - 37s 396ms/step - loss: 1.1739 - accuracy: 0.5783 - val_loss: 1.1901 - val_accuracy: 0.4723 - lr: 0.0010
Epoch 5/60
94/94 [===========